# Step 3 — Official-split UNSW-NB15 dataset audit

This notebook audits the **two user-provided Kaggle paths** without transforming or training on the data. Before the audit it validates the **full official class distribution** of each file and automatically corrects the train/test *roles* when the filenames are reversed. If the two distributions cannot be matched to the official partitions, it **stops** (rather than silently train on the wrong split). It writes only small aggregate artifacts under `data/manifests/unsw_nb15/`.

**Before running:** enable **Internet** in Kaggle settings to clone the public research repo and publish audit results. To auto-sync, add a Kaggle Notebook Secret named `pushRare` containing a **fine-grained GitHub token**, with **Contents: Read and write** limited to `itsCodeBakery/rare-attack-spatiotemporal-ids`. Enable access to that secret for this notebook. Never paste your token into the notebook or chat. Without the secret the audit runs but the GitHub push is skipped.

The audit is **not** proof that CSV row order reflects real traffic time. We will evaluate that before creating temporal windows.

In [ ]:
from pathlib import Path
import subprocess, os, sys
TRAIN=Path("/kaggle/input/datasets/primus11/unsw-nb15-dataset/unsw_train.csv")
TEST=Path("/kaggle/input/datasets/primus11/unsw-nb15-dataset/unsw_test.csv")
REPO=Path("/kaggle/working/rare-attack-spatiotemporal-ids")
print("Train present:",TRAIN.is_file(),"Test present:",TEST.is_file())
assert TRAIN.is_file() and TEST.is_file(), "Attach the dataset or correct its paths."

In [ ]:
url="https://github.com/itsCodeBakery/rare-attack-spatiotemporal-ids.git"
if REPO.is_dir():
    subprocess.run(["git","-C",str(REPO),"pull","--ff-only"],check=True)
else:
    subprocess.run(["git","clone","--depth","1",url,str(REPO)],check=True)
source_sha=subprocess.check_output(["git","-C",str(REPO),"rev-parse","HEAD"],text=True).strip()
print("Source Git commit:",source_sha)
from sys import path as _sys_path
if str(REPO / "src") not in _sys_path:
    _sys_path.insert(0, str(REPO / "src"))
from rareids.splits import resolve_by_class_counts
TRAIN, TEST, split_status, split_counts = resolve_by_class_counts(TRAIN, TEST)
print("OFFICIAL UNSW-NB15 split verified:", split_status)
print("Using training:", TRAIN)
print("Using testing: ", TEST)

In [ ]:
OUT=REPO/"data"/"manifests"/"unsw_nb15"
subprocess.run([sys.executable,str(REPO/"scripts"/"audit_unsw.py"),
               "--train",str(TRAIN),"--test",str(TEST),"--output",str(OUT)],
              cwd=str(REPO),check=True)
print((OUT/"audit_report.md").read_text())

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    token=UserSecretsClient().get_secret("pushRare")
except Exception:
    token=None
if token:
    environment=os.environ.copy()
    environment["GITHUB_TOKEN"]=token
    try:
        subprocess.run([sys.executable,str(REPO/"scripts"/"sync_manifest.py"),
                        "--directory",str(OUT)],cwd=str(REPO),
                        env=environment,check=True)
    finally:
        environment.pop("GITHUB_TOKEN",None)
        del token
else:
    print("NOT YET SYNCED: no accessible Kaggle pushRare secret.")
    print("Audit files remain locally at",OUT)

## Expected outcome

The report will show the **observed** row counts, feature count, schema ordering, attack-class distribution, duplicate fingerprints, timestamp-column candidates and warnings. The final cell prints a GitHub commit URL if publication succeeded. Share the report/commit URL before we proceed to model development.